# LAB | Programming Foundation Exercises | HomeGuard

## STEP 1

In [5]:
"""
HomeGuard Security System Simulator
Author: Carlos Martinez Boto
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.
"""
 
import random
from datetime import datetime

In [6]:
# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
 
# Current system state
current_mode = "AWAY"

## STEP 2

Define sensor creation function, create sensors for the Peterson home

In [7]:
def create_sensor(sensor_id, location, sensor_type, threshold=None):
    """
    Creates a sensor data structure.
    
    Parameters:
    - sensor_id: Unique identifier for the sensor
    - location: Where the sensor is located (e.g., "Living Room")
    - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
    - threshold: Optional threshold value for the sensor
    
    Returns:
    - A dictionary representing the sensor
    """
    sensor = {"id": sensor_id, "location": location, "type": sensor_type}
    if threshold != None:
        sensor["threshold"] = threshold
    
    return sensor

In [8]:
# Initialize sensors for the Peterson home
sensors = [
    create_sensor("LR_M", "living_room", "motion"), #motion for the living room 
    create_sensor("K_T", "kitchen", "temperature", 60), #temperature for the kitchen with threshold
    create_sensor("FD_D", "front_door", "door"), #door for the front door
    create_sensor("B_S", "bedroom", "smoke")  #smoke for the bedroom 
]

Checkpoint: verify that the sensors are created

In [9]:
print(f"Initialized {len(sensors)} sensors")
for sensor in sensors:
    print(f"  - {sensor['id']}: {sensor['location']} ({sensor['type']})")

Initialized 4 sensors
  - LR_M: living_room (motion)
  - K_T: kitchen (temperature)
  - FD_D: front_door (door)
  - B_S: bedroom (smoke)


Create alerts

In [10]:
def create_alert(severity, message, sensor_id, timestamp):
    """
    Creates an alert data structure.
    
    Parameters:
    - severity: Alert severity level (LOW, MEDIUM, HIGH, CRITICAL)
    - message: Description of the alert
    - sensor_id: ID of the sensor that triggered the alert
    - timestamp: When the alert was triggered
    
    Returns:
    - A dictionary representing the alert
    """
    alert = {"severity": severity, "message": message, "sensor_id": sensor_id, "timestamp": timestamp}
    return alert

## STEP 3

Conditional statements to check sensor thresholds and system modes.

In [11]:
def is_abnormal_reading(sensor, reading_value):
    """
    Checks if a sensor reading is abnormal based on sensor type and thresholds.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    
    Returns:
    - True if the reading is abnormal, False otherwise
    """
    sensor_type = sensor["type"]
    
    # Logic to check for abnormal readings
    if sensor_type == "temperature": # For temperature sensors: check if below 35°F or above 95°F
        if reading_value < 35 or reading_value > 95:
            return True
    elif sensor_type == "motion": # For motion sensors: check if motion is detected (reading_value == True)
        if reading_value:
            return True
    elif sensor_type == "door": # For door sensors: check if door is open (reading_value == "OPEN")
        if reading_value == "OPEN":
            return True
    elif sensor_type == "smoke": # For smoke sensors: check if smoke is detected (reading_value == "DETECTED")
        if reading_value == "DETECTED":
            return True

    return False

In [12]:
def is_outside_comfort_reading(sensor, reading_value):
    """
    Checks if a sensor reading is outside the comfort range.

    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor

    Returns:
    - True if the reading is outside the comfort range, False otherwise
    """
    
    # Temperature sensors: check if below 65°F or above 75°F
    sensor_type = sensor["type"]

    if sensor_type == "temperature":
        return reading_value < 65 or reading_value > 75
    else:
        return False

In [13]:
def should_trigger_security_alert(sensor, reading_value, system_mode):
    """
    Determines if a security alert should be triggered. Cases:
    - Motion detected when house is in "away" mode
    - Door opened when house is in "away" mode
    # - Multiple sensors triggered simultaneously (possible break-in)
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    - system_mode: Current system mode (HOME, AWAY, SLEEP)
    
    Returns:
    - True if a security alert should be triggered, False otherwise
    """

    sensor_type = sensor["type"]

    if system_mode == "AWAY":
        if sensor_type == "motion" and reading_value:
            return True
        elif sensor_type == "door" and reading_value == "OPEN":
            return True

    return False

Checkpoint: Test the functions

In [14]:
# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")  # Should be True
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")  # Should be False

34°F is abnormal: True
68°F is abnormal: False


In [15]:
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False

Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


## STEP 4

Function to generate realistic sensor readings

In [16]:
def generate_reading(sensor):
    """
    Generates a realistic reading for a sensor.
    
    Parameters:
    - sensor: Sensor dictionary
    
    Returns:
    - A realistic reading value based on sensor type (random values to simulate real sensor data)
    """
    sensor_type = sensor["type"]
    
    # Logic to check for abnormal readings
    if sensor_type == "temperature": # Temperature: random value between 30-100°F
        return random.randint(30, 100)
    elif sensor_type == "motion": # Motion: random boolean (True/False)
        return random.choice([True, False])
    elif sensor_type == "door": # Door: random choice between "OPEN" and "CLOSED"
        return random.choice(["OPEN", "CLOSED"])
    elif sensor_type == "smoke": # Smoke: random choice between "CLEAR" and "DETECTED"
        return random.choice(["CLEAR", "DETECTED"])

Function to process readings and determine if alerts are needed

In [17]:
def process_reading(sensor, reading_value, system_mode):
    """
    Processes a sensor reading and determines if alerts are needed.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The reading from the sensor
    - system_mode: Current system mode
    
    Returns:
    - A list of alerts (empty if no alerts needed)

    Severity policy:
    - CRITICAL -> intrusion or fire
    - HIGH -> overheating risk
    - MEDIUM -> freezing risk
    - LOW -> comfort

    """
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    sensor_id = sensor["id"]
    sensor_type = sensor["type"]
      
    # Security alerts
    # - Motion detected when house is in "AWAY" mode
    # - Door opened when house is in "AWAY" mode
    if should_trigger_security_alert(sensor, reading_value, system_mode):
        alerts.append(create_alert("CRITICAL", "SECURITY_ALERT", sensor_id, timestamp))

    # Safety alerts
    # - Temperature below 35°F (frozen pipe risk) or above 95°F (equipment failure)
    # - Smoke detected (fire risk)
    if sensor_type == "temperature":
        if reading_value < 35:
            alerts.append(create_alert("MEDIUM", "TEMP_LOW", sensor_id, timestamp))
        elif reading_value > 95:
            alerts.append(create_alert("HIGH", "TEMP_HIGH", sensor_id, timestamp))

    elif sensor_type == "smoke":
        if reading_value == "DETECTED":
            alerts.append(create_alert("CRITICAL", "SMOKE_DETECTED", sensor_id, timestamp))
    
    # Comfort notifications (only in HOME mode)
    # - Temperature outside comfort range (65-75°F) when home is in "home" mode
    if system_mode == "HOME" and sensor_type == "temperature":
        if is_outside_comfort_reading(sensor, reading_value):
            alerts.append(create_alert("LOW", "TEMP_COMFORT", sensor_id, timestamp))

    return alerts

Function to trigger and display alerts

In [18]:
def trigger_alert(alert):
    """
    Displays an alert to the user.
    
    Parameters:
    - alert: Alert dictionary
    """
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")

Logging function to record all events

In [19]:
def log_event(message, timestamp=None):
    """
    Logs an event to the console.
    
    Parameters:
    - message: The message to log
    - timestamp: Optional timestamp (uses current time if not provided)
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

Checkpoint: Test the functions

In [25]:
# Test reading generation
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")
 
# Test processing
alerts = process_reading(test_sensor, True, "AWAY")
# print(alerts)
if alerts:
    trigger_alert(alerts[0])

Generated reading for living_room: False
[ALERT!] 🔥 CRITICAL: SECURITY_ALERT


## STEP 5

Convert the sensor dictionary into a Sensor class with methods.

In [ ]:
class Sensor:
    """
    Represents a sensor in the HomeGuard system.
    """
    
    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        """
        Initializes a new sensor.
        
        Parameters:
        - sensor_id: Unique identifier for the sensor
        - location: Where the sensor is located
        - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
        - threshold: Optional threshold value for the sensor
        """
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None
 
    
    def read(self):
        """
        Generates and stores a new reading for this sensor.
        
        Returns:
        - The reading value
        """
        if self.type == "temperature": # Temperature: random value between 30-100°F
            reading = random.randint(30, 100)
        elif self.type == "motion": # Motion: random boolean (True/False)
            reading = random.choice([True, False])
        elif self.type == "door": # Door: random choice between "OPEN" and "CLOSED"
            reading = random.choice(["OPEN", "CLOSED"])
        elif self.type == "smoke": # Smoke: random choice between "CLEAR" and "DETECTED"
            reading = random.choice(["CLEAR", "DETECTED"])

        self.current_value = reading
        return reading
        
    def isAbnormal(self):
        """
        Checks if the current reading is abnormal.
        
        Returns:
        - True if the reading is abnormal, False otherwise
        """
        if self.type == "temperature": # For temperature sensors: check if below 35°F or above 95°F
            if self.current_value < 35 or self.current_value > 95:
                return True
        elif self.type == "motion": # For motion sensors: check if motion is detected (reading_value == True)
            if self.current_value:
                return True
        elif self.type == "door": # For door sensors: check if door is open (reading_value == "OPEN")
            if self.current_value == "OPEN":
                return True
        elif self.type == "smoke": # For smoke sensors: check if smoke is detected (reading_value == "DETECTED")
            if self.current_value == "DETECTED":
                return True
        
        return False
  
    
    def reset(self):
        """
        Resets the sensor's current reading to None.
        """
        # YOUR CODE HERE: Set current_value to None
        self.current_value = None
    
    def __str__(self):
        """
        Returns a string representation of the sensor.
        """
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"
 
# Create sensor objects using the class
sensor_objects = [
    Sensor("MOTION_001", "Living Room", "motion"),
    Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
    Sensor("DOOR_001", "Front Door", "door"),
    Sensor("SMOKE_001", "Bedroom", "smoke")
]

Checkpoint: Test your class:

In [22]:
# Create and test a sensor
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")

Sensor reading: 65
Is abnormal: False
Sensor info: TEST_001 (Test Room): 65


## STEP 6

Combine all components into a complete simulator that runs continuously.

In [ ]:
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    """
    Runs the HomeGuard security system simulation.
    
    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    # Use sensor objects instead of dictionaries
    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]
    
    # Simulate time passing (each iteration = 1 minute)
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")
        
        # Read all sensors
        for sensor in sensors:
            reading = sensor.read()
            
            # Display the reading
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            # Process the reading and trigger alerts if needed
            # Convert sensor object to dict format for process_reading function
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            # Trigger all alerts
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")
        
        # Small delay to make output readable (optional)
        import time
        time.sleep(0.5)  # 0.5 second delay between iterations
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)
 
# Main execution
if __name__ == "__main__":
    # Run the simulation
    run_simulation(duration_minutes=3, system_mode="AWAY")
    

=== HomeGuard Security System ===
Mode: AWAY


Time: 15:37:17
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 40°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🔥 CRITICAL: SECURITY_ALERT
[LOG] [15:37:17] Sending notification to homeowner...
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SMOKE_DETECTED
[LOG] [15:37:17] Sending notification to homeowner...



Time: 15:37:17
[READING] Living Room Motion: DETECTED
[ALERT!] 🔥 CRITICAL: SECURITY_ALERT
[LOG] [15:37:17] Sending notification to homeowner...
[READING] Kitchen Temperature: 61°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🔥 CRITICAL: SECURITY_ALERT
[LOG] [15:37:17] Sending notification to homeowner...
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SMOKE_DETECTED
[LOG] [15:37:17] Sending notification to homeowner...

Time: 15:37:18
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 93°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Simulation complete!
